# Nagelhus et al. (2023), distance tuning of single cells

**Paper.** Nagelhus et al. (2023), *Neuron* 111, 2091-2104,
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008); claims file written
from the version of record with its STAR Methods and Supplemental information.

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), version v1, the version
whose DOI the paper cites and the only one published (data check in `01_single_cells.ipynb`).

| Claim | Tested or blocked |
|---|---|
| fig6ab-single-cell-distance-tuning | tested |
| figs5l-without-positively-tuned | tested |
| fig6ef-least-tuned-cells | tested |
| figs8eh-least-tuned-smaller-datasets | tested |
| fig7-single-cell-properties | tested |
| fig7-slope-variant | tested |

The distance decoder runs with one cell at a time (Equation 1: distance tuning is accuracy in bin
1 minus bin 5). Significance comes from 100 circular shifts of the spike train in each trial, by
20 s to the trial length less 20 s, above the 99th or below the 1st percentile. Cell readings of
the claims file: (a) the 1,189 cells of the CA1 file, each in its session; (b) the 1,180 cells of
the 16 decoding sessions. The 897 cells of (b) that are also in the CA1 file are the same recordings,
with the same tracking, and give the same values; only the other 283 are computed again. Every run
uses the two speed readings and the two rate-map readings.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)

started = time.time()
RAW_POP = []
for p in oc.POP_FILES:
    RAW_POP += oc.load(p)[0]
CA1, _ = oc.load(oc.SINGLE_FILES["CA1"])
POP, replaced = oc.decoding_sessions(RAW_POP, CA1)
LINK = [oc.link_population(p, CA1) for p in RAW_POP]
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION}: read in {time.time() - started:.0f} s; tracking from the CA1 file for {replaced}")
READINGS = [(sp, rm) for sp in "ab" for rm in "ab"]

In [ ]:
rows = []
for k, p in enumerate(POP):
    cs, m = LINK[k]
    rows.append({"session": f"{p.rat}#{p.index}", "cells": p.n_units, "in CA1 file": len(m), "CA1 session": cs.index,
                 "same tracking": bool(np.array_equal(cs.trial("NO").t, p.trials[0].t) and np.allclose(cs.trial("NO").y, p.trials[0].y))})
print(pd.DataFrame(rows).to_string(index=False))
print("CA1 file: cells", sum(s.n_units for s in CA1), "in", len(CA1), "sessions")

In [ ]:
N_SHIFT = 100


def shifts_for(trA, trB, key):
    rng = np.random.default_rng(oc.seed_for("fig6ab-single-cell-distance-tuning", *key))
    TA, TB = trA.duration, trB.duration
    return np.r_[[[0.0, 0.0]], np.c_[rng.uniform(20, TA - 20, N_SHIFT), rng.uniform(20, TB - 20, N_SHIFT)]]


def tune_session(trA, trB, units, key, sp, rm):
    ctx = oc.PairContext(trA, trB, sp == "a")
    out = {}
    for u in units:
        acc = oc.single_cell_tuning(ctx, (u, u), rm, shifts_for(trA, trB, key + (u,)))
        out[u] = acc
    return out


TUNING = {}
t0 = time.time()
for sp, rm in READINGS:
    res_a = {}
    for si, s in enumerate(CA1):
        r = tune_session(s.trial("NO"), s.trial("O"), range(s.n_units), ("CA1", si), sp, rm)
        for u, acc in r.items():
            res_a["CA1", si, u] = acc
    res_b = {}
    for k, p in enumerate(POP):
        cs, m = LINK[k]
        todo = [u for u in range(p.n_units) if u not in m]
        r = tune_session(p.trials[0], p.trials[1], todo, ("POP", k), sp, rm) if todo else {}
        for u in range(p.n_units):
            res_b["POP", k, u] = res_a["CA1", cs.index, m[u]] if u in m else r[u]
    TUNING[sp, rm] = (res_a, res_b)
    print(f"speed {sp}, rm {rm}: {len(res_a)} + {len(res_b) - sum(len(LINK[k][1]) for k in range(len(POP)))} cells in {time.time() - t0:.0f} s")


def summarise(res):
    rows = []
    for key, acc in res.items():
        d = acc[:, 0] - acc[:, 4]
        sl = np.array([oc.slope(a) for a in acc])
        null = d[1:][np.isfinite(d[1:])]
        rows.append({"key": key, "tuning": d[0], "slope": sl[0],
                     "pos": d[0] > np.percentile(null, 99) if len(null) else False,
                     "neg": d[0] < np.percentile(null, 1) if len(null) else False,
                     "slope_null_lo": np.percentile(sl[1:], 1), "slope_null_hi": np.percentile(sl[1:], 99)})
    return pd.DataFrame(rows)


SUM = {k: (summarise(a), summarise(b)) for k, (a, b) in TUNING.items()}

## fig6ab-single-cell-distance-tuning

> Single cells decoded with almost flat accuracy across distance; their distance tuning (bin 1 minus
> bin 5 accuracy) averaged 0.02, 98.7% lay between -0.2 and 0.2, 12.2% (145) had significant positive
> and 3.8% (45) significant negative tuning.

In [ ]:
rows = []
for (sp, rm), (Sa, Sb) in SUM.items():
    for reading, S in (("a", Sa), ("b", Sb)):
        t = S.tuning.dropna()
        vals = {"mean": t.mean(), "within": np.mean(np.abs(t) <= 0.2), "pos": S.pos.mean(), "neg": S.neg.mean()}
        ok = {"mean": abs(vals["mean"] - 0.02) <= 0.005, "within": abs(vals["within"] - 0.987) <= 0.0987,
              "pos": abs(vals["pos"] - 0.122) <= 0.0122, "neg": abs(vals["neg"] - 0.038) <= 0.0038}
        rows.append({"speed": sp, "rm": rm, "cells": reading, "n": len(S), "no value": int(S.tuning.isna().sum()),
                     "mean tuning": vals["mean"], "within +/-0.2": vals["within"], "sig positive": vals["pos"],
                     "n positive": int(S.pos.sum()), "sig negative": vals["neg"], "n negative": int(S.neg.sum()),
                     **{f"{k} ok": v for k, v in ok.items()},
                     "outcome": "reproduced" if all(ok.values()) else ("partially" if ok["mean"] and ok["within"] else "not")})
fig6ab = pd.DataFrame(rows)
print(fig6ab.to_string(index=False))
print("\nPer speed and rm, best outcome over the two cell readings:")
order = {"reproduced": 2, "partially": 1, "not": 0}
print(fig6ab.assign(o=fig6ab.outcome.map(order)).groupby(["speed", "rm"]).o.max().map({v: k for k, v in order.items()}).to_string())
Sa = SUM["a", "a"][0]
print("\nDistance tuning, cells reading a, speed a, rm a: range", Sa.tuning.min().round(3), "to", Sa.tuning.max().round(3))
Sa["rat"] = [CA1[k[1]].rat for k in Sa.key]
print(Sa.groupby("rat").agg(cells=("tuning", "size"), mean=("tuning", "mean"), pos=("pos", "mean"), neg=("neg", "mean")).to_string())

**Result: partially reproduced.** Under every speed, rate-map and cell reading, three of the four
values are within tolerance. The mean distance tuning is 0.015 to 0.017 (0.02 ± 0.005), 97.7 to 98.7%
of cells lie between -0.2 and 0.2, and 11.9 to 12.7% have significant positive tuning (0.122 ± 0.0122).
The fraction with significant negative tuning is 4.2 to 5.6%, above 3.8 ± 0.38%. The criterion's
partial rule (mean and within-bounds fraction match, a significant fraction does not) applies. Mean
tuning is positive in every rat (0.008 to 0.041, printed above).

## figs5l-without-positively-tuned

> After removing the cells with significant positive distance tuning, decoding accuracy still fell
> gradually with distance.

In [ ]:
rows = []
for sp, rm in READINGS:
    Sb = SUM[sp, rm][1]
    pos = {k[1:] for k, v in zip(Sb.key, Sb.pos) if v}
    accs = []
    for k, p in enumerate(POP):
        keep = [u for u in range(p.n_units) if (k, u) not in pos]
        pA, pB = oc.half_parts(p.trials[0], sp == "a"), oc.half_parts(p.trials[1], sp == "a")
        accs.append(oc.decode_pair(pA, pB, "distance", obj=p.trials[1].obj, rm=rm, units=keep)["acc"])
    A = np.array(accs)
    h, pv = oc.kruskal(A.T)
    med = np.median(A, 0)
    rows.append({"speed": sp, "rm": rm, "cells removed": len(pos), "H": h, "p": pv,
                 **{f"median {i + 1}": med[i] for i in range(5)}, "bin 1 highest": int(np.argmax(med)) == 0})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (methods underspecified).** Without the 140 to 150 cells with significant positive
tuning, the Kruskal-Wallis test gives p ≤ 0.0001 under every reading (H = 25.0 to 29.5). Bin 1 has the
highest median under three readings, but under speed b, `rm` a bin 2 is highest (0.699 against
0.689), so the readings give different outcomes.

## fig6ef-least-tuned-cells

> Even the 20% of cells with the lowest distance tuning showed distance reorganization, with a bin 1
> minus bin 5 difference of 0.121 and a slope of -0.0267.

In [ ]:
KEEP = [1.0, 0.8, 0.6, 0.4, 0.2]


def least_tuned_curves(k, sp, rm):
    p = POP[k]
    Sb = SUM[sp, rm][1]
    t = np.array([Sb.tuning[Sb.key == ("POP", k, u)].iloc[0] for u in range(p.n_units)])
    order = np.argsort(np.nan_to_num(t, nan=np.inf))
    pA, pB = oc.half_parts(p.trials[0], sp == "a"), oc.half_parts(p.trials[1], sp == "a")
    out = []
    for f in KEEP:
        units = list(order[: max(1, int(round(f * p.n_units)))])
        acc = oc.decode_pair(pA, pB, "distance", obj=p.trials[1].obj, rm=rm, units=units)["acc"]
        out.append((f, len(units), acc[0] - acc[4], oc.slope(acc)))
    return out


big = [k for k, p in enumerate(POP) if p.n_units == 620][0]
rows = []
EF = {}
for sp, rm in READINGS:
    cur = least_tuned_curves(big, sp, rm)
    EF[sp, rm] = cur
    for f, n, d, sl in cur:
        rows.append({"speed": sp, "rm": rm, "kept": f, "cells": n, "bin1-bin5": d, "slope": sl,
                     "diff ok": abs(d - 0.121) <= 0.0121 if f == 0.2 else None,
                     "slope ok": abs(sl + 0.0267) <= 0.00267 if f == 0.2 else None})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** The 20% of the 620 cells with the lowest distance tuning (124 cells) give
a bin 1 minus bin 5 difference of -0.295 to -0.351 and a slope of +0.074 to +0.079 under every reading.
Accuracy rises with distance, against the paper's 0.121 and -0.0267. All 620 cells give 0.075 to
0.106 and -0.016 to -0.023. Keeping the lowest 80% already reverses the sign. Sorted by the claims
file's rule (lowest values first), the kept cells are those with negative tuning.

## figs8eh-least-tuned-smaller-datasets

> In the tetrode sessions and the 135-cell Neuropixels session too, keeping only the cells with the
> lowest distance tuning weakened but did not at first remove the distance reorganization; the slope
> reached zero when 60% (tetrode) or 50% (135-cell session) of the least-tuned cells were kept.

In [ ]:
rows = []
for sp, rm in READINGS:
    tet = [least_tuned_curves(k, sp, rm) for k, p in enumerate(POP) if p.technique == "tetrodes"]
    np135 = least_tuned_curves([k for k, p in enumerate(POP) if p.n_units == 135][0], sp, rm)
    for name, curves in (("tetrode (mean of 14)", tet), ("135-cell", [np135])):
        D = np.array([[c[2] for c in cur] for cur in curves])
        Sl = np.array([[c[3] for c in cur] for cur in curves])
        md, ms = np.nanmean(D, 0), np.nanmean(Sl, 0)
        zero = [f for f, s_ in zip(KEEP, ms) if s_ >= 0]
        target = 0.6 if name.startswith("tetrode") else 0.5
        rows.append({"speed": sp, "rm": rm, "dataset": name, **{f"diff {f}": d for f, d in zip(KEEP, md)},
                     **{f"slope {f}": s_ for f, s_ in zip(KEEP, ms)},
                     "largest kept with slope >= 0": max(zero) if zero else None,
                     "100% ok": md[0] > 0 and ms[0] < 0,
                     "shrink": bool(md[-1] < md[0] and ms[-1] > ms[0]),
                     "zero within tolerance": (abs(max(zero) - target) <= 0.2) if zero else False})
s8eh = pd.DataFrame(rows)
print(s8eh.to_string(index=False))

**Result: reproduced.** With all cells, the bin 1 minus bin 5 difference is positive and the slope
negative in both datasets (tetrode mean 0.178 to 0.188 and -0.043 to -0.045; 135-cell session 0.248
to 0.296 and -0.061 to -0.069). Both shrink as fewer of the least-tuned cells are kept. The largest
kept fraction with a mean slope of zero or above is 0.4 in both datasets under every reading, within
0.6 ± 0.2 and 0.5 ± 0.2. The 20% values are printed above (tetrode difference -0.073 to -0.091;
135-cell -0.12 to -0.18).

## fig7-single-cell-properties and fig7-slope-variant

> Single-cell properties barely predicted distance tuning; correlations with object-tuning score
> (r = 0.12), spatial information (r = -0.13), spatial correlation (r = -0.08) and rate overlap
> (r = -0.18) were significant but weak.

> Using the slope of accuracy against distance in place of distance tuning gave similar weak
> correlations with the same four properties.

Object-tuning score: the single score of the template matching (largest margin over its
threshold), under both width readings. Spatial information in bits per spike in the O trial's
2.5 cm map, with the trial's speed-filtered mean rate. NO-O spatial correlation and rate overlap
(readings a and b of the rate). Cell reading (a): the 1,189 cells of the CA1 file. Reading (b): the
897 decoding-session cells that are in the CA1 file (the others have no Object Moved trial, so no
object-tuning score).

In [ ]:
PROPS = {}
for rm in "ab":
    sc, ro_a, ro_b, si_ = {}, {}, {}, {}
    for i, s in enumerate(CA1):
        NO, O = s.trial("NO"), s.trial("O")
        mN, _ = oc.single_maps(NO, rm)
        mO, occO = oc.single_maps(O, rm)
        for u in range(s.n_units):
            sc[i, u] = oc.pearson_nan(mN[u], mO[u])
            ro_a[i, u] = oc.rate_overlap(oc.mean_rate(NO, u, True), oc.mean_rate(O, u, True))
            ro_b[i, u] = oc.rate_overlap(oc.mean_rate(NO, u, False), oc.mean_rate(O, u, False))
            si_[i, u] = oc.spatial_information(mO[u], occO, oc.mean_rate(O, u, True))
    for w in "ab":
        score = {(c["session"], c["unit"]): c["single"] for c in oc.object_tuning(CA1, rm, oc.WIDTHS[w], key="CA1")}
        PROPS[rm, w] = {"object-tuning score": score, "spatial information": si_, "spatial correlation": sc,
                        "rate overlap a": ro_a, "rate overlap b": ro_b}
linked = {(LINK[k][0].index, LINK[k][1][u]) for k, p in enumerate(POP) for u in LINK[k][1]}

EXPECT = {"tuning": {"object-tuning score": 0.12, "spatial information": -0.13, "spatial correlation": -0.08, "rate overlap": -0.18},
          "slope": {"object-tuning score": -0.12, "spatial information": 0.10, "spatial correlation": -0.08, "rate overlap": 0.19}}
rows = []
for sp, rm in READINGS:
    Sa = SUM[sp, rm][0]
    val = {"tuning": dict(zip([k[1:] for k in Sa.key], Sa.tuning)), "slope": dict(zip([k[1:] for k in Sa.key], Sa.slope))}
    for w in "ab":
        P = PROPS[rm, w]
        for ro in "ab":
            for cells in "ab":
                keys = [k for k in val["tuning"] if cells == "a" or k in linked]
                for measure in ("tuning", "slope"):
                    rec = {"speed": sp, "rm": rm, "w": w, "rate overlap": ro, "cells": cells, "measure": measure, "n": len(keys)}
                    allok = True
                    for prop in ("object-tuning score", "spatial information", "spatial correlation", "rate overlap"):
                        src = P[prop if prop != "rate overlap" else f"rate overlap {ro}"]
                        x = np.array([src[k] for k in keys])
                        y = np.array([val[measure][k] for k in keys])
                        ok = np.isfinite(x) & np.isfinite(y)
                        r, p = stats.pearsonr(x[ok], y[ok])
                        e = EXPECT[measure][prop]
                        good = abs(r - e) <= 0.05 and np.sign(r) == np.sign(e) and (p < 0.05 if measure == "tuning" else True) and (abs(r) < 0.3 if measure == "tuning" else True)
                        allok &= good
                        rec[f"r {prop}"] = r
                        rec[f"p {prop}"] = p
                    rec["all within"] = allok
                    rows.append(rec)
fig7 = pd.DataFrame(rows)
cols = ["speed", "rm", "w", "rate overlap", "cells", "measure", "n"] + [c for c in fig7.columns if c.startswith("r ")] + ["all within"]
print(fig7[cols].to_string(index=False))
print("\np-values, tuning:")
print(fig7[fig7.measure == "tuning"][["speed", "rm", "w", "rate overlap", "cells"] + [c for c in fig7.columns if c.startswith("p ")]].to_string(index=False))
print("\nReadings meeting the criterion, per speed and rm:")
print(fig7.groupby(["measure", "speed", "rm"])["all within"].any().to_string())
print("\nPer property, readings (of", int(len(fig7) / 2), "per measure) with r within 0.05 of the printed value and the same sign:")
for measure in ("tuning", "slope"):
    d = fig7[fig7.measure == measure]
    for prop, e in EXPECT[measure].items():
        good = (np.abs(d[f"r {prop}"] - e) <= 0.05) & (np.sign(d[f"r {prop}"]) == np.sign(e))
        sig = d[f"p {prop}"] < 0.05
        print(f"  {measure}, {prop} (printed {e}): within {int(good.sum())}, of which p < 0.05 {int((good & sig).sum())}; "
              f"r from {d[f'r {prop}'].min():.3f} to {d[f'r {prop}'].max():.3f}")
print(f"\nTotal time {time.time() - started:.0f} s")

**Result: blocked (methods underspecified)** (fig7-single-cell-properties). It depends on fig1c-object-tuned-fraction, which is blocked (methods underspecified), so it is blocked for the same reason; the result under every reading is given here. No reading meets the whole criterion.
Spatial information is not correlated with distance tuning: r = -0.023 to 0.010, p ≥ 0.43, against
the printed -0.13. The other three correlations are weak (|r| < 0.3) and mostly within 0.05 of the
printed values, with p < 0.05 for the 1,189 cells: object-tuning score r = 0.12 to 0.16; spatial
correlation r = -0.06 to -0.07, which is not significant under speed b, `rm` a; rate overlap r = -0.15
to -0.16. Counts per property are printed above.

**Result: blocked (methods underspecified)** (fig7-slope-variant). It depends on fig7-single-cell-properties, which is blocked. With the slope in place of distance tuning,
the object-tuning score (r = -0.12 to -0.16) and rate overlap (r = 0.15) are within 0.05 of the
printed values over the 1,189 cells. Spatial information (r = -0.02 to 0.01, printed 0.10) is not, and spatial
correlation has the opposite sign (r = +0.05 to +0.09, printed -0.08). The claims file notes that
this printed sign breaks the pattern of the other three.

## Summary

| Claim | Outcome |
|---|---|
| fig6ab-single-cell-distance-tuning | partially reproduced (significant negative fraction above tolerance) |
| figs5l-without-positively-tuned | blocked, methods underspecified (bin 1 highest under 3 of 4 readings) |
| fig6ef-least-tuned-cells | not reproduced (the least-tuned 20% decode better far from the object) |
| figs8eh-least-tuned-smaller-datasets | reproduced |
| fig7-single-cell-properties | blocked, depends on fig1c (partially reproduced under every reading: no correlation with spatial information) |
| fig7-slope-variant | blocked, depends on fig7 (partially reproduced under every reading) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the cells of the decoding sessions that are also in the
CA1 file take their values from it (same recording and tracking); object-tuning scores exist only for
cells with an Object Moved trial; spatial information uses the speed-filtered mean rate of the trial.

What was found about the dataset: 283 of the 425 tetrode cells of the decoding sessions cannot be
linked to the CA1 file, so they have no Object Moved trial and no object-tuning score.